# Übung: Daten und Vorverarbeitung mit Keras - Dos and Don'ts

Diese Übung kommt bewusst vor der Modellwahl-Übung. Der Schwerpunkt liegt auf Datenqualität und Vorverarbeitung: Welche Spalten dürfen ins Modell, welche Schritte müssen nur auf Trainingsdaten gelernt werden, und wo entsteht Data Leakage?

**Vorgegebener Datensatz:** Titanic aus den TensorFlow-Datasets-Tutorialdaten.

- Training: `https://storage.googleapis.com/tf-datasets/titanic/train.csv`
- Test: `https://storage.googleapis.com/tf-datasets/titanic/eval.csv`

**Wichtig:** Der Pipeline-Teil bleibt in Keras/TensorFlow. Nutzen Sie pandas nur zur Inspektion der Rohdaten. Keine scikit-learn-Pipeline in dieser Übung.

## Lernziele

- vorgegebenen Tabular-Datensatz laden und prüfen
- Wertebereiche, fehlende Werte, Zielverteilung und Auffälligkeiten systematisch untersuchen
- numerische Zusammenhänge und Korrelationen interpretieren
- kategorische Ausprägungen, Kardinalität und Zielraten analysieren
- numerische, kategorische und Zielspalten sauber trennen
- Train/Validation/Test und Data Leakage verstehen
- Keras-Preprocessing mit `Normalization` und `StringLookup` aufbauen
- `adapt(...)` nur auf Trainingsdaten ausführen
- unbekannte Kategorien und fehlende Werte robust behandeln
- Keras-Preprocessing-Pipeline speichern und wieder laden
- Dos and Don'ts für Datenvorverarbeitung formulieren


## Aufgabe 1 - Setup und vorgegebenen Datensatz laden

Laden Sie die beiden CSV-Dateien über `keras.utils.get_file(...)` und lesen Sie sie anschließend mit pandas ein.

Minimaler Hinweis:
- `train_url = "https://storage.googleapis.com/tf-datasets/titanic/train.csv"`
- `test_url = "https://storage.googleapis.com/tf-datasets/titanic/eval.csv"`
- `keras.utils.get_file("titanic_train.csv", train_url)` lädt die Datei in den lokalen Keras-Cache.
- Zielspalte ist `survived`.


**Auszufüllen:**

- Trainingszeilen: ____________________
- Testzeilen: ____________________
- Spaltennamen:
- Zielspalte: ____________________
- Welche Spalte enthält das Label und darf nicht als Input verwendet werden?

Antwort:


## Aufgabe 2 - Rohdaten inspizieren

Untersuchen Sie die Rohdaten, bevor Sie irgendeine Vorverarbeitung bauen.

Prüfen Sie:
- `.head()`
- `.info()`
- `.describe(include="all")`
- fehlende Werte pro Spalte
- eindeutige Werte in kategorischen Spalten
- Klassenverteilung von `survived`

Minimaler Hinweis:
- Dokumentieren Sie Beobachtungen, nicht nur Code-Ausgaben.


**Auszufüllen:**

| Spalte | Typ | Beobachtung / mögliches Problem |
| --- | --- | --- |
| sex |  |  |
| age |  |  |
| n_siblings_spouses |  |  |
| parch |  |  |
| fare |  |  |
| class |  |  |
| deck |  |  |
| embark_town |  |  |
| alone |  |  |
| survived |  |  |


## Aufgabe 3 - Missing Values, Wertebereiche und Plausibilität

Untersuchen Sie genauer, wo Werte fehlen und ob Wertebereiche plausibel wirken.

Erstellen Sie mindestens:
- Tabelle mit fehlenden Werten absolut und prozentual pro Spalte
- Vergleich fehlender Werte zwischen Trainings- und Testdatei
- Histogramme oder Boxplots für `age`, `fare`, `n_siblings_spouses` und `parch`
- Prüfung von Minimum, Maximum, Median und Quantilen für numerische Features
- kurze Einschätzung, ob fehlende Werte zufällig wirken oder mit anderen Spalten zusammenhängen könnten

Minimaler Hinweis:
- Fehlende Werte sind nicht automatisch Fehler.
- Ein Feature mit vielen fehlenden Werten kann trotzdem nützlich sein, wenn das Fehlen selbst Information trägt.
- Prüfen Sie `deck` besonders bewusst.


**Auszufüllen:**

| Spalte | Missing train absolut | Missing train % | Missing test % | Entscheidung / Beobachtung |
| --- | ---: | ---: | ---: | --- |
| age |  |  |  |  |
| fare |  |  |  |  |
| deck |  |  |  |  |
| embark_town |  |  |  |  |

- Welche numerischen Wertebereiche wirken plausibel?
- Welche Wertebereiche sollten vor dem Modellieren noch fachlich geprüft werden?
- Gibt es Spalten, bei denen Missingness selbst ein Signal sein könnte?

Antwort:


## Aufgabe 4 - Numerische Zusammenhänge und Korrelationen

Untersuchen Sie, welche numerischen Features miteinander und mit `survived` zusammenhängen.

Erstellen Sie mindestens:
- Korrelationsmatrix für numerische Features plus `survived`
- Heatmap der Korrelationen
- sortierte Korrelationen der numerischen Features mit `survived`
- Scatterplots oder gruppierte Verteilungen für auffällige Paare, z. B. `age` vs. `fare`
- Mittelwert/Median numerischer Features getrennt nach `survived`

Minimaler Hinweis:
- `survived` ist binär. Eine Korrelation mit `survived` kann als grober linearer Zielbezug gelesen werden, ersetzt aber keine Modellbewertung.
- Niedrige Korrelation bedeutet nicht automatisch, dass ein Feature nutzlos ist. Nichtlineare oder kategorische Effekte können trotzdem wichtig sein.


**Auszufüllen:**

| Numerisches Feature | Korrelation mit `survived` | Beobachtung |
| --- | ---: | --- |
| age |  |  |
| n_siblings_spouses |  |  |
| parch |  |  |
| fare |  |  |

- Welche numerischen Features hängen am stärksten mit dem Target zusammen?
- Welche numerischen Features korrelieren untereinander?
- Welche Korrelationen könnten fachlich plausibel sein?
- Wo reicht Korrelation als Erklärung nicht aus?

Antwort:


## Aufgabe 5 - Kategorische Features verstehen

Untersuchen Sie die kategorischen Spalten gründlich, bevor Sie sie encoden.

Erstellen Sie mindestens:
- Anzahl eindeutiger Werte pro kategorischer Spalte
- Häufigkeitstabelle je Kategorie
- relative Häufigkeiten je Kategorie
- Überlebensrate je Kategorie
- Markierung seltener Kategorien, z. B. Kategorien mit weniger als 10 Samples
- Vergleich, ob Kategorien im Test vorkommen, die im Training selten oder nicht vorhanden sind

Minimaler Hinweis:
- Hohe Kardinalität kann One-Hot-Encoding groß machen.
- Sehr seltene Kategorien können instabile Zielraten haben.
- Zielraten sind hilfreich zur Datenexploration, dürfen aber nicht direkt als gelerntes Encoding aus allen Daten übernommen werden.


**Auszufüllen:**

| Kategorisches Feature | Anzahl Kategorien train | seltene Kategorien? | stärkster sichtbarer Target-Bezug |
| --- | ---: | --- | --- |
| sex |  |  |  |
| class |  |  |  |
| deck |  |  |  |
| embark_town |  |  |  |
| alone |  |  |  |

- Welche kategorische Spalte scheint besonders informativ?
- Welche Kategorie-Zielraten sind wegen kleiner Fallzahl unsicher?
- Gibt es Kategorien in Test, die im Training nicht vorkommen?
- Warum muss die spätere `StringLookup`-Schicht mit unbekannten Kategorien umgehen können?

Antwort:


## Aufgabe 6 - Feature-Rollen festlegen

Legen Sie fest, welche Spalten numerisch und welche kategorisch verarbeitet werden.

Empfehlung für diese Übung:
- numerisch: `age`, `n_siblings_spouses`, `parch`, `fare`
- kategorisch: `sex`, `class`, `deck`, `embark_town`, `alone`
- Target: `survived`

Minimaler Hinweis:
- Nutzen Sie diese Listen später zentral in allen Funktionen.
- Das Target darf nie in den Input-Dictionarys landen.
- Begründen Sie Ihre Auswahl mit den Beobachtungen aus den Aufgaben 2 bis 5.


**Auszufüllen:**

Numerische Features:

Kategorische Features:

Target:

Warum ist diese Trennung für Keras-Preprocessing wichtig?

Antwort:


## Aufgabe 7 - Validation-Split aus Training erzeugen

Der Datensatz liefert bereits eine Testdatei. Erzeugen Sie aus der Trainingsdatei zusätzlich einen Validierungssplit.

Empfehlung:
- Training intern: 80 Prozent der Trainingsdatei
- Validierung: 20 Prozent der Trainingsdatei
- Test: unveränderte `eval.csv`
- möglichst ähnliche `survived`-Verteilung in Training und Validierung

Minimaler Hinweis:
- Wenn Sie ohne scikit-learn arbeiten, können Sie mit pandas mischen und dann nach Klassenanteilen prüfen.
- Der Testdatensatz bleibt bis zum Schluss unberührt.


**Auszufüllen:**

| Split | Zeilen | Überlebensrate |
| --- | ---: | ---: |
| Training |  |  |
| Validierung |  |  |
| Test |  |  |

Warum wird Test nicht zur Pipeline- oder Modellwahl benutzt?

Antwort:


## Aufgabe 8 - `tf.data.Dataset` aus DataFrames bauen

Schreiben Sie eine Funktion `dataframe_to_dataset(...)`, die Rohdaten in ein Keras-kompatibles Dataset umwandelt.

Anforderungen:
- `survived` als Label abtrennen
- numerische Features als `float32`
- kategorische Features als Strings
- fehlende kategorische Werte als `"[MISSING]"`
- Training shuffeln, Validierung und Test nicht shuffeln
- batchen und prefetch nutzen

Minimaler Hinweis:
- Keras kann Feature-Dictionaries verarbeiten: `{feature_name: tensor}`.
- Pro Feature soll ein Tensor mit Shape `(batch, 1)` entstehen.


**Auszufüllen:**

- Batch Size: ____________________
- Feature-Keys im Batch:
- Shape eines numerischen Feature-Batches: ____________________
- Shape eines kategorischen Feature-Batches: ____________________
- Shape des Label-Batches: ____________________

Antwort:


## Aufgabe 9 - Keras-Inputs für Rohdaten definieren

Definieren Sie einen `keras.Input` pro Feature.

Anforderungen:
- numerische Features: `dtype=tf.float32`, `shape=(1,)`
- kategorische Features: `dtype=tf.string`, `shape=(1,)`
- Input-Namen müssen exakt den Spaltennamen entsprechen

Minimaler Hinweis:
- Das Modell soll später Rohdaten-Dictionaries akzeptieren.
- Dadurch bleibt die Pipeline näher an echten Produktionsdaten.


**Auszufüllen:**

- Anzahl Inputs: ____________________
- Numerische Input-Namen:
- Kategorische Input-Namen:
- Warum ist ein Input pro Rohspalte hier hilfreich?

Antwort:


## Aufgabe 10 - Numerische Features mit Keras normalisieren

Bauen Sie für jedes numerische Feature eine `layers.Normalization`-Schicht.

Anforderungen:
- `adapt(...)` nur mit Trainingsdaten ausführen
- keine Validation- oder Testdaten in `adapt(...)` verwenden
- normalisierte numerische Features sammeln

Minimaler Hinweis:
- `normalizer.adapt(train_df[feature].to_numpy().reshape(-1, 1))`
- Falls eine numerische Spalte fehlende Werte enthält, entscheiden Sie zuerst eine einfache Keras-nahe Strategie, z. B. vor dem Dataset-Bau mit Trainingsmedian auffüllen und diesen Wert dokumentieren.


**Auszufüllen:**

| Feature | gelernter Mittelwert nur aus Training | gelernte Varianz/Std. | Besonderheit |
| --- | ---: | ---: | --- |
| age |  |  |  |
| n_siblings_spouses |  |  |  |
| parch |  |  |  |
| fare |  |  |  |

Was wäre an `adapt(...)` auf allen Daten falsch?

Antwort:


## Aufgabe 11 - Kategorische Features mit Keras encoden

Bauen Sie für jedes kategorische Feature eine `layers.StringLookup(output_mode="one_hot")`-Schicht.

Anforderungen:
- Vokabular nur aus Trainingsdaten lernen
- fehlende Werte als `[MISSING]` behandeln
- unbekannte Kategorien robust abfangen
- encodierte Features sammeln

Minimaler Hinweis:
- `lookup.adapt(train_df[feature].fillna("[MISSING]").astype(str).to_numpy())`
- `StringLookup` hat standardmäßig einen OOV-Bucket für unbekannte Kategorien.


**Auszufüllen:**

| Feature | Anzahl gelernter Kategorien | Umgang mit unbekannten Kategorien |
| --- | ---: | --- |
| sex |  |  |
| class |  |  |
| deck |  |  |
| embark_town |  |  |
| alone |  |  |

Warum ist ein OOV-Bucket wichtig?

Antwort:


## Aufgabe 12 - Preprocessing-Pipeline als Keras-Modell bauen

Verbinden Sie alle normalisierten und encodierten Features mit `layers.Concatenate` und bauen Sie daraus ein eigenes `preprocessing_model`.

Anforderungen:
- Inputs: Rohdaten-Inputs aus Aufgabe 9
- Output: ein numerischer Feature-Vektor
- keine Dense-Klassifikationsschichten in diesem Modell
- `preprocessing_model.summary()` anzeigen

Minimaler Hinweis:
- Dieses Modell ist die reine Vorverarbeitungspipeline.
- Es kann separat getestet, gespeichert und geladen werden.


**Auszufüllen:**

- Output-Shape der Preprocessing-Pipeline: ____________________
- Welche Schritte sind im Modell enthalten?
- Welche Schritte sind nicht im Modell enthalten?
- Warum ist eine speicherbare Preprocessing-Pipeline besser als lose Notebook-Zellen?

Antwort:


## Aufgabe 13 - Preprocessing-Pipeline testen

Wenden Sie `preprocessing_model` auf einen Batch aus `train_ds` und einen Batch aus `val_ds` an.

Prüfen Sie:
- Ausgabe ist numerisch
- Batch-Dimension bleibt erhalten
- gleiche Pipeline funktioniert für Training und Validierung
- keine Fehlermeldung bei kategorischen Strings

Minimaler Hinweis:
- `features_batch, labels_batch = next(iter(train_ds))`
- `preprocessed = preprocessing_model(features_batch)`


**Auszufüllen:**

- Shape nach Preprocessing für Training: ____________________
- Shape nach Preprocessing für Validierung: ____________________
- Gibt es NaN-Werte im Output?
- Welche Fehler würden hier auf falsche Shapes oder dtypes hindeuten?

Antwort:


## Aufgabe 14 - Pipeline speichern und wieder laden

Speichern Sie `preprocessing_model` als `.keras`-Datei und laden Sie es wieder.

Anforderungen:
- speichern mit `preprocessing_model.save(...)`
- laden mit `keras.models.load_model(...)`
- denselben Batch durch Original und geladenes Modell schicken
- Outputs numerisch vergleichen

Minimaler Hinweis:
- Nutzen Sie einen lokalen Ordner wie `generated_data/20_preprocessing_pipeline` oder einen temporären Pfad.
- Vergleichen Sie z. B. mit `np.allclose(...)`.


**Auszufüllen:**

- Speicherpfad: ____________________
- Laden erfolgreich? ____________________
- Stimmen Original- und Loaded-Output überein? ____________________
- Warum ist dieser Test wichtig?

Antwort:


## Aufgabe 15 - Klassifikationsmodell auf die Pipeline setzen

Bauen Sie ein einfaches Keras-Modell, das die Preprocessing-Pipeline nutzt und danach binär klassifiziert.

Empfehlung:
- `x = preprocessing_model(inputs)`
- ein oder zwei Dense-Layer mit ReLU
- Output: `Dense(1, activation="sigmoid")`
- Loss: `BinaryCrossentropy`
- Metriken: Accuracy, Precision, Recall, AUC

Minimaler Hinweis:
- Der Fokus bleibt Preprocessing. Trainieren Sie kurz und interpretieren Sie nicht zu viel in die Modellgüte hinein.


**Auszufüllen:**

| Metrik | Training | Validierung |
| --- | ---: | ---: |
| Loss |  |  |
| Accuracy |  |  |
| Precision |  |  |
| Recall |  |  |
| AUC |  |  |

Was sagt dieses Ergebnis über die Pipeline aus, und was noch nicht?

Antwort:


## Aufgabe 16 - Leakage-Fälle bewusst erkennen

Beschreiben Sie für jeden Fall, warum er falsch wäre und wie Sie ihn vermeiden.

Fälle:
- `adapt(...)` auf Training plus Validation plus Test
- `survived` versehentlich als Input verwenden
- Testdaten mehrfach zur Modellwahl nutzen
- Kategorien aus Testdaten ins Vokabular übernehmen
- fehlende Werte auf Basis des gesamten Datensatzes imputieren


**Auszufüllen:**

| Don't | Warum falsch? | Korrekte Variante |
| --- | --- | --- |
| `adapt` auf allen Daten |  |  |
| Target als Input |  |  |
| Test zur Modellwahl |  |  |
| Test-Kategorien im Vokabular |  |  |
| Imputation aus allen Daten |  |  |


## Aufgabe 17 - Dos and Don'ts als Abschluss-Checkliste

Formulieren Sie Ihre eigene Checkliste für zukünftige Keras-Projekte mit Tabular-Daten.

**Auszufüllen:**

| Do | Warum? |
| --- | --- |
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |

| Don't | Warum nicht? |
| --- | --- |
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |

Kurze Abschlussantwort:

- Welcher Vorverarbeitungsfehler wäre in echten Projekten am gefährlichsten?
- Welche Keras-Preprocessing-Layer haben Sie genutzt?
- Was muss gespeichert werden, damit neue Rohdaten später korrekt verarbeitet werden?
